# Sonic Transaction History Analysis

This notebook performs a range of analyses on historic transaction data
collected from the Sonic main-net.

## 1. Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
# Replace '../tx_data.csv.gz' with the path to your CSV file
data = pd.read_csv('../tx_data.csv.gz')

# clean up column names
data.columns = data.columns.str.strip()

## 2. Transaction Rate
The following charts illustrate the number of transactions per day.


In [ ]:
# Convert timestamp to datetime and extract date
dates = pd.to_datetime(data['timestamp'], unit='s').dt.date

# Count transactions per day
tx_per_day = dates.value_counts().sort_index()

# Plot
plt.figure(figsize=(14, 6))
tx_per_day.plot()
plt.title('Number of Transactions per Day')
plt.xlabel('Date')
plt.ylabel('Number of Transactions')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate total gas usage per day
gas_per_day = data.groupby(dates)['gas_used'].sum()

# Plot total gas usage per day
plt.figure(figsize=(14, 6))
gas_per_day.plot()
plt.title('Total Gas Usage per Day')
plt.xlabel('Date')
plt.ylabel('Total Gas Used')
plt.tight_layout()
plt.show()

In [ ]:
# Ensure gas_used and gas_price are of dtype 'object' to avoid overflow
data['gas_used_obj'] = data['gas_used'].astype(object)
data['gas_price_obj'] = data['gas_price'].astype(object)

# Calculate daily sum of gas_used * gas_price, scaled down by 1e18
daily_gas_fee_safe = data.groupby(dates).apply(lambda x: (x['gas_used_obj'] * x['gas_price_obj']).sum() / 1e18)

# Plot the daily gas fee
plt.figure(figsize=(14, 6))
daily_gas_fee_safe.plot()
plt.title('Total Gas Fee per Day')
plt.xlabel('Date')
plt.ylabel('Total Gas Fee (S)')
plt.tight_layout()
plt.show()

In [ ]:
# Compute CDF
sorted_fees = np.sort(daily_gas_fee_safe.values)
cdf = np.arange(1, len(sorted_fees) + 1) / len(sorted_fees)

plt.figure(figsize=(10, 6))
plt.plot(sorted_fees, cdf, label='CDF of Daily Gas Fees')
plt.xlabel('Daily Gas Fee (S)')
plt.ylabel('CDF')
plt.title('CDF of Daily Gas Fees')

# Mark the 50% (median) case
median_fee = np.median(sorted_fees)
plt.axvline(median_fee, color='red', linestyle='--', label='50% (Median)')
plt.axhline(0.5, color='red', linestyle='--')
plt.text(median_fee, 0.52, f'Median: {median_fee:.2f}', color='red', ha='left')

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Convert index to PeriodIndex (monthly)
monthly_index = pd.PeriodIndex(daily_gas_fee_safe.index, freq='M')
monthly_fees = daily_gas_fee_safe.groupby(monthly_index)

plt.figure(figsize=(12, 8))
for month, fees in monthly_fees:
    sorted_fees = np.sort(fees.values)
    cdf = np.arange(1, len(sorted_fees) + 1) / len(sorted_fees)
    plt.plot(sorted_fees, cdf, label=str(month))

plt.xlabel('Daily Gas Fee (S)')
plt.ylabel('CDF')
plt.title('CDF of Daily Gas Fees per Month')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Group daily gas fee by month and calculate the average per month
avg_gas_fee_per_month = daily_gas_fee_safe.groupby(monthly_index).mean()

# Plot the average gas fee per month
plt.figure(figsize=(12, 6))
avg_gas_fee_per_month.plot(kind='bar')
plt.title('Average Gas Fee per Day by Month')
plt.xlabel('Month')
plt.ylabel('Average Gas Fee (S)')
plt.tight_layout()
plt.show()

In [ ]:
# Display the average gas fee per month as a table
avg_gas_fee_per_month.to_frame(name='Average Gas Fee (S)')

In [ ]:
# Group daily gas used by month and calculate the average per month
avg_gas_used_per_month = gas_per_day.groupby(monthly_index).mean()

# Plot the average gas used per day by month
plt.figure(figsize=(12, 6))
avg_gas_used_per_month.plot(kind='bar')
plt.title('Average Gas Used per Day by Month')
plt.xlabel('Month')
plt.ylabel('Average Gas Used')
plt.tight_layout()
plt.show()

In [ ]:
# Display the average gas fee per month as a table
avg_gas_used_per_month.to_frame(name='Average Gas Used')

## 3. Gas Prices

The following charts plot a summary of the encountered gas prices.

In [ ]:
# Calculate average gas price per day
avg_gas_price_per_day = data.groupby(dates)['gas_price'].mean()

# Plot the average gas price per day
plt.figure(figsize=(14, 6))
avg_gas_price_per_day.plot()
plt.title('Average Gas Price per Day')
plt.xlabel('Date')
plt.ylabel('Average Gas Price')
plt.tight_layout()
plt.show()

In [ ]:
# Calculate percentiles for gas prices per day
percentiles = [0.5, 0.9, 0.99]
gas_price_percentiles = data.groupby(dates)['gas_price'].quantile(percentiles).unstack() / 1e9  # Convert from wei to Gwei

# Plot
plt.figure(figsize=(14, 6))
for p in percentiles:
    plt.plot(gas_price_percentiles.index, gas_price_percentiles[p], label=f'p{int(p*100)}')
plt.plot(avg_gas_price_per_day.index, avg_gas_price_per_day.values / 1e9, label='Average', linestyle='--')  # Convert from wei to Gwei
plt.title('Gas Price Percentiles per Day')
plt.xlabel('Date')
plt.ylabel('Gas Price (Gwei)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
avg_gas_price_per_day.plot()
plt.title('Average Gas Price per Day')
plt.xlabel('Date')
plt.ylabel('Average Gas Price')
plt.tight_layout()
plt.show()

In [ ]:
# Get the last day in the dataset
last_day = gas_price_percentiles.index[-1]

# Extract p50, p90, p99 for the last day
last_day_gas_prices = gas_price_percentiles.loc[last_day, [0.50, 0.90, 0.99]]

# Display as a table
last_day_gas_prices.to_frame(name='Gas Price (Gwei)')

In [ ]:
# Calculate percentiles for base fees per day
percentiles = [0.5, 0.9, 0.99]
base_fee_percentiles = data.groupby(dates)['base_fee'].quantile(percentiles).unstack() / 1e9  # Convert from wei to Gwei

# Plot
plt.figure(figsize=(14, 6))
for p in percentiles:
    plt.plot(base_fee_percentiles.index, base_fee_percentiles[p], label=f'p{int(p*100)}')
plt.title('Base Fee Percentiles per Day')
plt.xlabel('Date')
plt.ylabel('Base Fee (Gwei)')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Transaction Gas Usage

In [ ]:
# Calculate desired percentiles and max for gas_used per day
percentiles = [0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
gas_used_percentiles = data.groupby(dates)['gas_used'].quantile(percentiles).unstack()

plt.figure(figsize=(14, 7))
for p in percentiles:
    plt.plot(gas_used_percentiles.index, gas_used_percentiles[p], label=f'p{int(p*100)}')
plt.title('Transaction Gas Usage Percentiles per Day')
plt.xlabel('Date')
plt.ylabel('Gas Used')
plt.legend()
plt.tight_layout()
plt.show()